# Impact of instance-dependent normalization in the Sherrington–Kirkpatrick model

Comparisons of the **spectral-gap distributions at $\beta=4$**. All figures read `sk_variants/spectral_gaps_models_merged.hdf5` and use the existing `/move/model/n/idx` datasets. No instances or gaps are recomputed.

## 1. Imports, styling, and exports

All plot preferences are set in the following code cell. Figures are **18 cm wide and 11 cm high**, sized for the upper half of a portrait A4 page with room for a caption. Axis labels and titles use 11 pt; tick labels and legends use 10 pt. The two rows share their horizontal axes.

Figures are saved to `plots/` in PDF, PNG, and PGF formats. LaTeX rendering uses `pdflatex`. Re-run all cells to overwrite the previous exports. Include the figures at their native width or at approximately `18cm`; further shrinking also shrinks the text.


In [7]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# All styling, layout, and export preferences are configured here.
REGULAR_SIZE = 11
SMALL_SIZE = 10
SAVE_FIGURES = True
OUTPUT_DIR = Path("plots")
EXPORT_FORMATS = ("pdf", "png", "pgf")
EXPORT_BACKEND = "pgf"  # Consistent LaTeX rendering; retain the notebook display backend.

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "",
    "font.family": "serif",
    "font.size": REGULAR_SIZE,
    "axes.labelsize": REGULAR_SIZE,
    "axes.titlesize": REGULAR_SIZE,
    "figure.titlesize": REGULAR_SIZE,
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "legend.fontsize": SMALL_SIZE,
    "axes.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "legend.frameon": False,
    "savefig.bbox": None,  # Preserve the intended physical page size.
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
})

COLORS = {
    "normalized": "#222222",
    "unnormalized": "#999999",
    "unnormalized_sk": "#004488",
    "new_sk": "#EE7733",
    "deterministic_normalized": "#AA3377",
}
MOVE_LABELS = {"local1": "Local spin-flip", "uniform": "Uniform", "layden": "Quantum enhanced"}

# Size for the upper half of a portrait A4 page, leaving room for a caption.
# Export at the intended print size so fonts are not halved on inclusion.
FIGURE_WIDTH_CM = 18.0
FIGURE_HEIGHT_CM = 11.0
FIGURE_STYLE = dict(nrows=2, ncols=3,
                    figsize=(FIGURE_WIDTH_CM / 2.54, FIGURE_HEIGHT_CM / 2.54),
                    sharex="col", sharey="row")
ROW_LABELS = (r"Spectral gap $\delta$" + "\n(distribution)",
              r"Spectral gap $\delta$" + "\n" + r"(mean $\pm$ std)")
MODEL_OFFSET = 0.25
VIOLIN_WIDTH_TWO_MODELS = 0.40
VIOLIN_WIDTH_DEFAULT = 0.30
X_MARGIN = 0.65
POINT_STYLE = dict(s=20, zorder=4)
VIOLIN_BODY_STYLE = dict(alpha=0.4, linewidth=0.9)
VIOLIN_MEDIAN_STYLE = dict(linewidth=1.4)
ERRORBAR_STYLE = dict(marker="o", markersize=4.0, linewidth=1.4, capsize=2, zorder=3)
LEGEND_PATCH_STYLE = dict(alpha=0.65)
LEGEND_STYLE = dict(loc="lower center", bbox_to_anchor=(0.5, 0.015), columnspacing=1.2,
                    handlelength=1.3, handletextpad=0.5)
PANEL_SPACING = dict(left=0.14, right=0.99, top=0.92, bottom=0.22,
                     wspace=0.14, hspace=0.32)


def save_figure(fig, basename):
    """Save a comparison to plots/ as PDF, PNG, and PGF."""
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        for ext in EXPORT_FORMATS:
            fig.savefig(OUTPUT_DIR / f"{basename}.{ext}", backend=EXPORT_BACKEND)


## 2. Load merged spectral gaps

In [8]:
BETA, N_INSTANCES = 4.0, 100
MOVES = ("local1", "uniform", "layden")
NS = range(3, 11)
MODELS = {
    "normalized": "Instance-normalized",
    "unnormalized": "Unnormalized",
    "unnormalized_sk": "Converted random-field SK",
    "new_sk": "New random-field SK",
    "deterministic_normalized": "Deterministically normalized",
}
FILE = Path("sk_variants/spectral_gaps_models_merged.hdf5")
if not FILE.is_file():
    raise FileNotFoundError(f"Missing {FILE.resolve()}; run 'python spectral_gaps_models.py collide' first.")

spectral_gaps = {move: {model: {} for model in MODELS} for move in MOVES}
with h5py.File(FILE, "r") as f:
    if "beta" in f.attrs and not np.isclose(float(f.attrs["beta"]), BETA):
        raise ValueError(f"Expected beta={BETA}, found beta={f.attrs['beta']}")
    for move in MOVES:
        for model in MODELS:
            for n in NS:
                values = np.full(N_INSTANCES, np.nan)
                path = f"{move}/{model}/{n}"
                if path in f:
                    for idx, dataset in f[path].items():
                        if idx.isdigit() and 0 <= int(idx) < N_INSTANCES:
                            values[int(idx)] = dataset[()]
                spectral_gaps[move][model][n] = values

print(f"Loaded {FILE} | beta={BETA:g} | n={min(NS)}..{max(NS)}")
for move in MOVES:
    counts = {model: sum(np.isfinite(spectral_gaps[move][model][n]).sum() for n in NS)
              for model in MODELS}
    print(f"{move:8s}: " + ", ".join(f"{model}={count}" for model, count in counts.items()))

Loaded sk_variants/spectral_gaps_models_merged.hdf5 | beta=4 | n=3..10
local1  : normalized=800, unnormalized=800, unnormalized_sk=800, new_sk=800, deterministic_normalized=800
uniform : normalized=800, unnormalized=800, unnormalized_sk=800, new_sk=800, deterministic_normalized=800
layden  : normalized=800, unnormalized=800, unnormalized_sk=800, new_sk=800, deterministic_normalized=800


## 3. Plotting function

Each comparison has **violin distributions** in the first row and **arithmetic mean ± standard deviation** in the second row. Columns are the three proposal moves; each row shares its logarithmic spectral-gap axis across moves. When mean − standard deviation is nonpositive, the lower whisker is omitted on the log axis.

The plotting function uses the style settings and shared save function defined above.


In [9]:
def plot_comparison(models, basename):
    fig, axes = plt.subplots(**FIGURE_STYLE)
    offsets = np.linspace(-MODEL_OFFSET, MODEL_OFFSET, len(models)) if len(models) > 1 else [0]
    width = VIOLIN_WIDTH_TWO_MODELS if len(models) == 2 else VIOLIN_WIDTH_DEFAULT

    for col, move in enumerate(MOVES):
        violin_ax, stats_ax = axes[:, col]
        violin_ax.set_title(MOVE_LABELS[move])
        for offset, model in zip(offsets, models):
            color = COLORS[model]
            means, stds = [], []
            for n in NS:
                values = spectral_gaps[move][model][n]
                values = values[np.isfinite(values) & (values > 0) & (values <= 1 + 1e-8)]
                if len(values):
                    pos = n + offset
                    if len(values) < 2 or np.allclose(values, values[0], rtol=1e-8, atol=0):
                        violin_ax.scatter(pos, np.median(values), color=color, **POINT_STYLE)
                    else:
                        violin = violin_ax.violinplot([values], positions=[pos],
                            widths=width, showmedians=True, showextrema=False)
                        violin["bodies"][0].set(facecolor=color, edgecolor=color,
                            **VIOLIN_BODY_STYLE)
                        violin["cmedians"].set(color=color, **VIOLIN_MEDIAN_STYLE)
                means.append(np.mean(values) if len(values) else np.nan)
                stds.append(np.std(values, ddof=1) if len(values) > 1 else 0.0)

            means, stds = np.array(means), np.array(stds)
            valid = np.isfinite(means) & (means > 0)
            lower = np.where(means > stds, stds, 0.0)
            stats_ax.errorbar(np.array(list(NS))[valid] + offset, means[valid],
                yerr=[lower[valid], stds[valid]], color=color, **ERRORBAR_STYLE)

        for ax in (violin_ax, stats_ax):
            ax.set_yscale("log")
            ax.set_xticks(list(NS))
            ax.set_xlim(min(NS) - X_MARGIN, max(NS) + X_MARGIN)
        stats_ax.set_xlabel(r"$n$")
        if col == 0:
            violin_ax.set_ylabel(ROW_LABELS[0])
            stats_ax.set_ylabel(ROW_LABELS[1])

    fig.legend([Patch(facecolor=COLORS[m], edgecolor=COLORS[m], **LEGEND_PATCH_STYLE) for m in models],
               [MODELS[m] for m in models], ncol=len(models), **LEGEND_STYLE)
    fig.subplots_adjust(**PANEL_SPACING)
    save_figure(fig, basename)
    plt.show()


## 4. Instance-dependent versus deterministic normalization

Both use the same initial Gaussian coefficients. The comparison isolates the instance-specific factor $\alpha$ versus the fixed $\alpha_0=\sqrt{2/(n+1)}$.

In [10]:
plot_comparison(("normalized", "deterministic_normalized"), "model_normalization")


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)

! LaTeX Error: File `type1ec.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.8 \usepackage
               [utf8]{inputenc}^^M
No pages of output.
Transcript written on file.log.




<Figure size 708.661x433.071 with 6 Axes>

## 5. Normalized versus random-field SK

Compare the original normalized ensemble with the converted and independently sampled random-field SK ensembles (`local_fields=True`). **Both coupling scale and local-field scale differ**, so this does not isolate fields alone.

In [11]:
plot_comparison(("normalized", "unnormalized_sk", "new_sk"), "model_fields")


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)

! LaTeX Error: File `type1ec.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.8 \usepackage
               [utf8]{inputenc}^^M
No pages of output.
Transcript written on file.log.




<Figure size 708.661x433.071 with 6 Axes>

## 6. Normalized versus unnormalized Gaussian instances

Same initial coefficients, but at the same numerical $\beta=4$ the unnormalized ensemble is effectively colder; the target Gibbs distributions differ.

In [12]:
plot_comparison(("normalized", "unnormalized"), "model_unnormalized")


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./file.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)

! LaTeX Error: File `type1ec.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.8 \usepackage
               [utf8]{inputenc}^^M
No pages of output.
Transcript written on file.log.




<Figure size 708.661x433.071 with 6 Axes>

## 7. Diagnostics

Copyable counts and medians, by move, model, and $n$. `valid` counts finite positive gaps no greater than one.

In [13]:
for move in MOVES:
    print(f"\n=== {MOVE_LABELS[move]} | beta={BETA:g} ===")
    print(f"{'n':>2}  {'model':<27} {'valid':>5} {'missing':>7} {'invalid':>7} {'median':>12} {'q25':>12} {'q75':>12}")
    for n in NS:
        for model in MODELS:
            v = spectral_gaps[move][model][n]
            finite = v[np.isfinite(v)]
            good = finite[(finite > 0) & (finite <= 1 + 1e-8)]
            q25, med, q75 = np.percentile(good, [25, 50, 75]) if good.size else [np.nan] * 3
            print(f"{n:>2}  {model:<27} {good.size:>5} {np.sum(~np.isfinite(v)):>7} "
                  f"{finite.size - good.size:>7} {med:>12.4g} {q25:>12.4g} {q75:>12.4g}")


=== Local spin-flip | beta=4 ===
 n  model                       valid missing invalid       median          q25          q75
 3  normalized                    100       0       0       0.0223     0.002471       0.2298
 3  unnormalized                  100       0       0      0.01622    0.0006649       0.2385
 3  unnormalized_sk               100       0       0       0.2225      0.03516       0.3256
 3  new_sk                        100       0       0       0.2139      0.02643        0.318
 3  deterministic_normalized      100       0       0      0.03692     0.003376         0.22
 4  normalized                    100       0       0     0.004283    0.0002142      0.01974
 4  unnormalized                  100       0       0    0.0007616    8.644e-06      0.01077
 4  unnormalized_sk               100       0       0      0.08671      0.01465       0.2067
 4  new_sk                        100       0       0       0.1089     0.008414       0.2293
 4  deterministic_normalized      10